In [ ]:
import os
import glob
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import numpy as np
import sys
root = os.path.abspath('../../..')
sys.path.append(root)

In [ ]:
def plot_traces_vs_time_fast(
    path_sec_files,
    save_html=False,
    output_dir=None,
    file_pattern="*.csv",
    vp_column='Vertical Position m',
    date_column='Date',
    time_hms_column='Time (HH:mm:ss)',
    time_frac_column='Time (Fract. Sec)',
    output_time_column='time_seconds',
    title='Vertical Position vs Time'
):
    """
    Plot Vertical Position as a function of cumulative elapsed time.

    Logic
    -----
    - Read Date + HH:mm:ss and optionally fractional seconds.
    - Detect whether Time uses AM/PM.
    - Detect whether Date is MM/DD/YYYY or DD/MM/YYYY.
    - Normalize Date internally to MM/DD/YYYY.
    - If AM/PM is present, parse as 12-hour format and sort by Date + Time
      sequentially across multiple days.
    - Otherwise, keep 24-hour logic.
    - Sort by:
        1) parsed Date + Time
        2) fractional seconds (ascending), if available
    - Convert datetime to elapsed seconds from the first row.
    - Add fractional seconds if present.
    - Save final elapsed time into output_time_column.
    - Print first 10 rows for debugging.
    - Plot Vertical Position vs time_seconds.
    - Reverse y-axis.
    """

    if save_html:
        if output_dir is None:
            raise ValueError("If save_html=True, you must provide output_dir.")
        os.makedirs(output_dir, exist_ok=True)

    pattern = os.path.join(path_sec_files, file_pattern)
    matched_files = sorted(glob.glob(pattern))

    if not matched_files:
        raise FileNotFoundError(f"No CSV files were found with pattern: {pattern}")

    palette = px.colors.qualitative.Alphabet
    color_map = {
        os.path.splitext(os.path.basename(file_path))[0]: palette[i % len(palette)]
        for i, file_path in enumerate(matched_files)
    }

    fig = go.Figure()

    for file_path in matched_files:
        trace_name = os.path.splitext(os.path.basename(file_path))[0]
        color = color_map[trace_name]

        try:
            df = pd.read_csv(file_path)
        except Exception as e:
            print(f"[WARNING] Could not read {file_path}: {e}")
            continue

        required_cols = [vp_column, date_column, time_hms_column]
        missing_required = [c for c in required_cols if c not in df.columns]
        if missing_required:
            print(f"[WARNING] Missing required columns in {file_path}: {missing_required}")
            continue

        has_fractional_seconds = time_frac_column in df.columns

        # -------------------------
        # Basic cleaning
        # -------------------------
        df[vp_column] = pd.to_numeric(df[vp_column], errors='coerce')
        df[date_column] = df[date_column].astype(str).str.strip()
        df[time_hms_column] = df[time_hms_column].astype(str).str.strip()

        if has_fractional_seconds:
            df[time_frac_column] = pd.to_numeric(df[time_frac_column], errors='coerce')
            df = df.dropna(subset=[vp_column, date_column, time_hms_column, time_frac_column]).copy()
        else:
            df = df.dropna(subset=[vp_column, date_column, time_hms_column]).copy()

        if df.empty:
            print(f"[WARNING] {os.path.basename(file_path)} is empty after cleaning.")
            continue

        # -------------------------
        # Fractional seconds
        # -------------------------
        if has_fractional_seconds:
            df['_frac_seconds'] = df[time_frac_column].astype(float)
        else:
            df['_frac_seconds'] = 0.0

        # -------------------------
        # Detect whether Time uses AM/PM
        # -------------------------
        time_upper = df[time_hms_column].str.upper()
        uses_am_pm = time_upper.str.contains(r'\bAM\b|\bPM\b', regex=True, na=False).any()

        # -------------------------
        # Detect date format:
        # choose between MM/DD/YYYY and DD/MM/YYYY
        # -------------------------
        dates_raw = df[date_column].copy()

        parsed_mdy = pd.to_datetime(
            dates_raw,
            format='%m/%d/%Y',
            errors='coerce'
        )

        parsed_dmy = pd.to_datetime(
            dates_raw,
            format='%d/%m/%Y',
            errors='coerce'
        )

        valid_mdy = parsed_mdy.notna().sum()
        valid_dmy = parsed_dmy.notna().sum()

        if valid_dmy > valid_mdy:
            df['_date_parsed'] = parsed_dmy
            detected_date_format = 'DD/MM/YYYY'
        else:
            df['_date_parsed'] = parsed_mdy
            detected_date_format = 'MM/DD/YYYY'

        # Fallback general si todavía quedan fechas inválidas
        invalid_date_mask = df['_date_parsed'].isna()
        if invalid_date_mask.any():
            fallback_dates = pd.to_datetime(
                dates_raw[invalid_date_mask],
                errors='coerce',
                dayfirst=(detected_date_format == 'DD/MM/YYYY')
            )
            df.loc[invalid_date_mask, '_date_parsed'] = fallback_dates

        df = df.dropna(subset=['_date_parsed']).copy()

        if df.empty:
            print(f"[WARNING] No valid dates found in {os.path.basename(file_path)}.")
            continue

        # Normalize date to MM/DD/YYYY
        df['_date_normalized'] = df['_date_parsed'].dt.strftime('%m/%d/%Y')

        print(f"\n[INFO] {trace_name}: detected date format = {detected_date_format}. "
              f"Normalized internally to MM/DD/YYYY.")

        if uses_am_pm:
            print(f"[INFO] {trace_name}: AM/PM detected in '{time_hms_column}'. "
                  f"Using sequential ordering by Date + 12-hour Time.")

        # -------------------------
        # Build datetime from normalized Date + Time
        # -------------------------
        datetime_str = df['_date_normalized'] + ' ' + df[time_hms_column]

        if uses_am_pm:
            df['_datetime'] = pd.to_datetime(
                datetime_str,
                errors='coerce',
                format='%m/%d/%Y %I:%M:%S %p'
            )

            invalid_mask = df['_datetime'].isna()
            if invalid_mask.any():
                df.loc[invalid_mask, '_datetime'] = pd.to_datetime(
                    datetime_str[invalid_mask],
                    errors='coerce'
                )
        else:
            df['_datetime'] = pd.to_datetime(
                datetime_str,
                errors='coerce',
                format='%m/%d/%Y %H:%M:%S'
            )

            invalid_mask = df['_datetime'].isna()
            if invalid_mask.any():
                df.loc[invalid_mask, '_datetime'] = pd.to_datetime(
                    datetime_str[invalid_mask],
                    errors='coerce'
                )

        df = df.dropna(subset=['_datetime']).copy()

        if df.empty:
            print(f"[WARNING] No valid Date + Time values found in {os.path.basename(file_path)}.")
            continue

        # -------------------------
        # Sort correctly:
        # first by datetime, then by fractional seconds
        # -------------------------
        df = df.sort_values(
            by=['_datetime', '_frac_seconds'],
            ascending=[True, True]
        ).reset_index(drop=True)

        # -------------------------
        # Build elapsed seconds from first datetime
        # -------------------------
        t0_datetime = df['_datetime'].iloc[0]
        df['_elapsed_datetime_seconds'] = (
            df['_datetime'] - t0_datetime
        ).dt.total_seconds()

        df[output_time_column] = (
            df['_elapsed_datetime_seconds'] + df['_frac_seconds']
        ).astype(float)

        # -------------------------
        # Diagnostic delta
        # -------------------------
        df['_delta_t'] = df[output_time_column].diff()
        if not df.empty:
            df.loc[df.index[0], '_delta_t'] = 0.0

        # -------------------------
        # Debug print
        # -------------------------
        debug_cols = [date_column, '_date_normalized', time_hms_column]
        if has_fractional_seconds:
            debug_cols.append(time_frac_column)
        debug_cols += [
            '_datetime',
            '_frac_seconds',
            '_elapsed_datetime_seconds',
            output_time_column,
            '_delta_t'
        ]

        print(f"\n[DEBUG] Profile: {trace_name}")
        print(f"[DEBUG] First 10 rows after sorting and cumulative time construction:")
        print(df[debug_cols].head(10).to_string(index=False))
        print(f"[DEBUG] time range: min={df[output_time_column].min():.3f} s, max={df[output_time_column].max():.3f} s")

        # -------------------------
        # Arrays for plot
        # -------------------------
        x = df[output_time_column].to_numpy(dtype=np.float32)
        y = df[vp_column].to_numpy(dtype=np.float32)

        hover_template = (
            'time=%{x:.3f} s<br>VP=%{y:.3f}<extra></extra>'
            if has_fractional_seconds
            else 'time=%{x:.0f} s<br>VP=%{y:.3f}<extra></extra>'
        )

        fig.add_trace(
            go.Scattergl(
                x=x,
                y=y,
                mode='markers',
                name=trace_name,
                marker=dict(
                    color=color,
                    size=3
                ),
                hovertemplate=hover_template
            )
        )

    fig.update_layout(
        title=title,
        xaxis_title='Time (s)',
        yaxis_title=vp_column,
        template='plotly_white',
        legend_title='Trace',
        hovermode='closest',
        height=600
    )

    fig.update_yaxes(autorange='reversed')

    if save_html:
        folder_name = os.path.basename(os.path.normpath(path_sec_files))
        output_html = os.path.join(output_dir, f"{folder_name}_vs_time.html")

        fig.write_html(
            output_html,
            include_plotlyjs='cdn',
            full_html=True
        )
        print(f"[OK] HTML saved to: {output_html}")

    return fig

def debug_datetime_sequence(
    path_sec_files,
    save_html=False,
    output_dir=None,
    file_pattern="*.csv",
    date_column='Date',
    time_hms_column='Time (HH:mm:ss)',
    time_frac_column='Time (Fract. Sec)',
    title_prefix='Datetime sequence debug'
):
    """
    Debug temporal ordering in CSV traces.

    For each file:
    - builds a datetime from Date + Time (HH:mm:ss)
    - if fractional seconds exist, adds them as timedeltas
    - sorts by datetime_full
    - computes delta_t_seconds between consecutive rows
    - creates a plot:
        x = sample index
        y = datetime_full
    - hover shows raw time fields and delta_t_seconds

    This is meant to detect:
    - temporal jumps
    - backward steps
    - duplicated timestamps
    - bad parsing/order issues
    """

    if save_html:
        if output_dir is None:
            raise ValueError("If save_html=True, you must provide output_dir.")
        os.makedirs(output_dir, exist_ok=True)

    pattern = os.path.join(path_sec_files, file_pattern)
    matched_files = sorted(glob.glob(pattern))

    if not matched_files:
        raise FileNotFoundError(f"No CSV files were found with pattern: {pattern}")

    figures = {}

    for file_path in matched_files:
        trace_name = os.path.splitext(os.path.basename(file_path))[0]

        try:
            df = pd.read_csv(file_path)
        except Exception as e:
            print(f"[WARNING] Could not read {file_path}: {e}")
            continue

        required_cols = [date_column, time_hms_column]
        missing_required = [c for c in required_cols if c not in df.columns]
        if missing_required:
            print(f"[WARNING] Missing required columns in {file_path}: {missing_required}")
            continue

        has_fractional_seconds = time_frac_column in df.columns

        df[date_column] = df[date_column].astype(str).str.strip()
        df[time_hms_column] = df[time_hms_column].astype(str).str.strip()

        if has_fractional_seconds:
            df[time_frac_column] = pd.to_numeric(df[time_frac_column], errors='coerce')
            df = df.dropna(subset=[date_column, time_hms_column, time_frac_column]).copy()
        else:
            df = df.dropna(subset=[date_column, time_hms_column]).copy()
            df[time_frac_column] = 0.0

        if df.empty:
            print(f"[WARNING] {trace_name} is empty after cleaning.")
            continue

        # Try common date formats
        datetime_str = df[date_column] + ' ' + df[time_hms_column]

        dt = pd.to_datetime(datetime_str, format='%m/%d/%Y %H:%M:%S', errors='coerce')

        invalid_mask = dt.isna()
        if invalid_mask.any():
            dt_alt = pd.to_datetime(datetime_str[invalid_mask], format='%d/%m/%Y %H:%M:%S', errors='coerce')
            dt.loc[invalid_mask] = dt_alt

        invalid_mask = dt.isna()
        if invalid_mask.any():
            dt_fallback = pd.to_datetime(datetime_str[invalid_mask], errors='coerce')
            dt.loc[invalid_mask] = dt_fallback

        df['_datetime_base'] = dt
        df = df.dropna(subset=['_datetime_base']).copy()

        if df.empty:
            print(f"[WARNING] No valid datetime values found in {trace_name}.")
            continue

        df['_frac_seconds'] = pd.to_numeric(df[time_frac_column], errors='coerce').fillna(0.0)
        df['_datetime_full'] = df['_datetime_base'] + pd.to_timedelta(df['_frac_seconds'], unit='s')

        # Sort by full datetime
        df = df.sort_values(by='_datetime_full', ascending=True).reset_index(drop=True)

        # Diagnostics
        df['_sample_index'] = np.arange(len(df))
        df['_delta_t_seconds'] = df['_datetime_full'].diff().dt.total_seconds()
        if len(df) > 0:
            df.loc[0, '_delta_t_seconds'] = 0.0

        # Flag suspicious jumps/backward issues after sorting
        # backward should not happen after sorting, but duplicates/jumps can still be detected
        df['_is_duplicate_time'] = df['_delta_t_seconds'] == 0
        df['_is_large_jump'] = df['_delta_t_seconds'] > 2.0  # adjust threshold if needed

        print(f"\n[DEBUG] Profile: {trace_name}")
        print(df[
            [date_column, time_hms_column, time_frac_column, '_datetime_full', '_delta_t_seconds']
        ].head(15).to_string(index=False))
        print(
            f"[DEBUG] delta_t range: min={df['_delta_t_seconds'].min():.6f} s, "
            f"max={df['_delta_t_seconds'].max():.6f} s"
        )
        print(
            f"[DEBUG] duplicate timestamps: {int(df['_is_duplicate_time'].sum())}, "
            f"large jumps (>2 s): {int(df['_is_large_jump'].sum())}"
        )

        customdata = np.stack(
            [
                df[date_column].astype(str).to_numpy(),
                df[time_hms_column].astype(str).to_numpy(),
                df[time_frac_column].astype(float).to_numpy(),
                df['_delta_t_seconds'].astype(float).to_numpy(),
            ],
            axis=1
        )

        fig = go.Figure()

        fig.add_trace(
            go.Scattergl(
                x=df['_sample_index'],
                y=df['_datetime_full'],
                mode='markers',
                name=trace_name,
                customdata=customdata,
                marker=dict(size=4),
                hovertemplate=(
                    "index=%{x}<br>"
                    "datetime=%{y|%Y-%m-%d %H:%M:%S.%L}<br>"
                    "Date=%{customdata[0]}<br>"
                    "Time=%{customdata[1]}<br>"
                    "FracSec=%{customdata[2]:.6f}<br>"
                    "delta_t=%{customdata[3]:.6f} s"
                    "<extra></extra>"
                )
            )
        )

        # Optional: mark large jumps
        jumps = df[df['_is_large_jump']]
        if not jumps.empty:
            jump_customdata = np.stack(
                [
                    jumps[date_column].astype(str).to_numpy(),
                    jumps[time_hms_column].astype(str).to_numpy(),
                    jumps[time_frac_column].astype(float).to_numpy(),
                    jumps['_delta_t_seconds'].astype(float).to_numpy(),
                ],
                axis=1
            )

            fig.add_trace(
                go.Scattergl(
                    x=jumps['_sample_index'],
                    y=jumps['_datetime_full'],
                    mode='markers',
                    name='Large jumps',
                    customdata=jump_customdata,
                    marker=dict(size=8, symbol='x'),
                    hovertemplate=(
                        "index=%{x}<br>"
                        "datetime=%{y|%Y-%m-%d %H:%M:%S.%L}<br>"
                        "Date=%{customdata[0]}<br>"
                        "Time=%{customdata[1]}<br>"
                        "FracSec=%{customdata[2]:.6f}<br>"
                        "delta_t=%{customdata[3]:.6f} s"
                        "<extra></extra>"
                    )
                )
            )

        fig.update_layout(
            title=f"{title_prefix} - {trace_name}",
            xaxis_title='Sample index (after sorting)',
            yaxis_title='Datetime',
            template='plotly_white',
            hovermode='closest',
            height=650
        )

        figures[trace_name] = fig

        if save_html:
            output_html = os.path.join(output_dir, f"{trace_name}_datetime_debug.html")
            fig.write_html(output_html, include_plotlyjs='cdn', full_html=True)
            print(f"[OK] HTML saved to: {output_html}")

    return figures

In [ ]:
output_dir = f'{root}/results/html_time'
path_sec_files = f'{root}/data/raw/raw_aw6'

fig = plot_traces_vs_time_fast(
    path_sec_files,
    save_html=True,
    output_dir=output_dir,
    file_pattern="*.csv",
    vp_column='Vertical Position m',
    time_hms_column='Time (HH:mm:ss)',
    time_frac_column='Time (Fract. Sec)',
    title='Vertical Position vs Time'
    )

fig.show()

In [ ]:
figs = debug_datetime_sequence(
    path_sec_files=r"C:\Users\Mariana\Documents\freshwater_lens\data\raw\raw_lrs69",
    file_pattern="LRS69_D_YSI_20251203_fix.csv"
    
)
figs["LRS69_D_YSI_20251203_fix"].show()